# Шаг 10. Исправление библиотек для браузера (папка vendor)

В браузере не загрузились файлы ONNX Runtime Web: обычные сборки `*.min.mjs` ссылаются на пакет `onnxruntime-common` по имени, а браузер так не умеет — нужны самодостаточные сборки `*.bundle.min.mjs`. Ноутбук собирает папку `vendor` заново и **проверяет каждый файл**: в нём не должно остаться ссылок на пакеты по имени. Если такие ссылки есть в Transformers.js, они заменяются на путь к файлу ONNX Runtime внутри расширения.

GPU не нужен. Время: ≈2 минуты. **Прислать:** вывод ячейки 2. Архив `vendor_fixed.zip` с Диска — скачать.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, shutil, json
WORK = '/content/drive/MyDrive/ai-text-detector'
os.makedirs('/content/npm', exist_ok=True)
%cd /content/npm
!npm init -y > /dev/null 2>&1
!npm install --ignore-scripts --no-audit --no-fund @huggingface/transformers@3 onnxruntime-web@1 2>&1 | tail -1
!node -e "console.log('transformers', require('./node_modules/@huggingface/transformers/package.json').version, '| onnxruntime-web', require('./node_modules/onnxruntime-web/package.json').version)"

In [ ]:
STATIC = [re.compile(r'(?:^|[;}\s])import\s*(?:[\w$*{}\s,]+?\s*from\s*)?["\']([^"\'\s]+)["\']'),
          re.compile(r'(?:^|[;}\s])export\s*(?:\*|\{[^}]*\})\s*(?:as\s+[\w$]+\s*)?from\s*["\']([^"\'\s]+)["\']')]
DYNAMIC = re.compile(r'\bimport\s*\(\s*["\']([^"\'\s]+)["\']')
PKG = re.compile(r'^(?:@[\w.-]+/)?[\w.-]+(?:/[\w./-]+)?$')
def _bare(names): return sorted({m for m in names if PKG.match(m) and not m.startswith(('.', '/', 'node:'))})
def bare_imports(path):
    """Статические ссылки на модули по имени пакета — из-за них модуль не загрузится в браузере."""
    txt = open(path, encoding='utf-8', errors='ignore').read()
    return _bare([m for rx in STATIC for m in rx.findall(txt)])
def bare_dynamic(path):
    """Динамические import("пакет") — загружаются только при вызове (обычно в ветках для Node.js), браузеру не мешают."""
    return _bare(DYNAMIC.findall(open(path, encoding='utf-8', errors='ignore').read()))
odist = 'node_modules/onnxruntime-web/dist'; tdist = 'node_modules/@huggingface/transformers/dist'
out = '/content/vendor_fixed'; shutil.rmtree(out, ignore_errors=True)
os.makedirs(f'{out}/ort'); os.makedirs(f'{out}/transformers')
for f in glob.glob(f'{odist}/ort*.mjs') + glob.glob(f'{odist}/ort-wasm*'):
    if not f.endswith('.map'): shutil.copy(f, f'{out}/ort/')
print('Файлы ONNX Runtime Web и ссылки по имени пакета в них:')
for f in sorted(os.listdir(f'{out}/ort')):
    if f.endswith('.mjs'): print(f'  {f:40s}', bare_imports(f'{out}/ort/{f}') or 'нет — файл самодостаточен')
need = ['ort.wasm.bundle.min.mjs', 'ort.webgpu.bundle.min.mjs']
assert all(os.path.exists(f'{out}/ort/{f}') and not bare_imports(f'{out}/ort/{f}') for f in need), 'Нет самодостаточных сборок ONNX Runtime — пришли вывод мне'
print('\nTransformers.js:')
cands = {f: bare_imports(f'{tdist}/{f}') for f in ['transformers.min.js', 'transformers.web.min.js'] if os.path.exists(f'{tdist}/{f}')}
for f, b in cands.items(): print(f'  {f:28s} статические: {b or "нет"}; динамические (не мешают): {bare_dynamic(f"{tdist}/{f}") or "нет"}')
src = min(cands, key=lambda f: len(cands[f]))
txt = open(f'{tdist}/{src}', encoding='utf-8').read()
for spec in set(cands[src]) | set(bare_dynamic(f'{tdist}/{src}')):
    if spec.startswith('onnxruntime'):
        txt = re.sub(r'(["\'])' + re.escape(spec) + r'\1', '"../ort/ort.webgpu.bundle.min.mjs"', txt)
open(f'{out}/transformers/transformers.min.js', 'w', encoding='utf-8').write(txt)
left = bare_imports(f'{out}/transformers/transformers.min.js')
print(f'Взят {src}; после замены ссылок по имени осталось: {left or "нет"}')
assert not left, 'Остались неразрешимые ссылки — пришли вывод мне'
if os.path.exists(f'{WORK}/vendor_fixed.zip'): os.remove(f'{WORK}/vendor_fixed.zip')
shutil.make_archive(f'{WORK}/vendor_fixed', 'zip', '/content', 'vendor_fixed')
print('\nГотово: vendor_fixed.zip на Диске,', round(os.path.getsize(f'{WORK}/vendor_fixed.zip') / 2**20), 'МБ')